# Cardiac Patient Monitoring System

## 06 — Clustering and PCA

## Objective

Explore the structure of the cleaned patient feature space using PCA for dimensionality reduction and K-Means clustering. This is exploratory and does not create clinical patient groups.

## Imports and Load Data

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

DATA_PATH = DATA_DIR / "cardio_train.csv"

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv(DATA_DIR / "cardio_clean.csv")
X = df.drop(columns=["cardio"])


## Prepare Features for Unsupervised Learning

In [ ]:
# Standardize features so variables measured on different scales do not dominate distance calculations.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled matrix shape:", X_scaled.shape)


## PCA — Explained Variance

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8,4))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Cumulative Explained Variance")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_explained_variance.png", dpi=150)
plt.show()

components_90 = int(np.argmax(cumulative_variance >= 0.90) + 1)
print("Components needed for at least 90% cumulative variance:", components_90)


## PCA to Two Dimensions

In [ ]:
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

print("Explained variance ratio:", pca_2d.explained_variance_ratio_)
print("Total explained variance:", pca_2d.explained_variance_ratio_.sum())

plt.figure(figsize=(7,5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=8, alpha=0.35)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Patients Projected onto the First Two Principal Components")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_2d.png", dpi=150)
plt.show()


## K-Means Clustering

In [ ]:
cluster_scores = []

for k in range(2, 7):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    cluster_scores.append({"k": k, "silhouette_score": score})

cluster_scores = pd.DataFrame(cluster_scores)
display(cluster_scores)

best_k = int(cluster_scores.loc[cluster_scores["silhouette_score"].idxmax(), "k"])
print("Selected k by highest silhouette score:", best_k)


## Visualize Clusters in PCA Space

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8,5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, s=8, alpha=0.4)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-Means Clusters Visualized with PCA (k={best_k})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "kmeans_pca_clusters.png", dpi=150)
plt.show()

cluster_summary = pd.DataFrame({
    "cluster": labels
}).value_counts().sort_index().rename("count").to_frame()
display(cluster_summary)


## Interpretation

PCA compresses the feature space into a smaller number of components while preserving as much variance as possible. K-Means groups observations according to distance in the standardized feature space. The clusters should be interpreted as mathematical patterns in this dataset, not as medical diagnoses or clinically validated patient categories.

## Unsupervised Learning Conclusion

The project now includes dimensionality reduction and clustering, satisfying the unsupervised-analysis requirement. The final README should summarize the supervised and unsupervised findings together.